# HiGLUE Diagonal Integration of Human PFC Data

In this tutorial we outline the full HiGLUE pipeline to integrate snm3C-seq single-cell contact maps in human prefrontal cortex with scRNA-seq expression profiles from independent cells:

* snm3C-seq Human PFC: https://www.nature.com/articles/s41592-019-0547-z

* scRNA-seq Human PFC: https://autism.cells.ucsc.edu/

The scRNA-seq data should be in AnnData format and must contain a `counts` layer with the raw gene count matrix and a `celltype` obs column (note that it can just be a single arbitrary value representing "unknown")

In [1]:
import scanpy as sc

rna = sc.read('/mnt/jinstore/JinLab01/dmp131/scRNA/brain_autism_PFC_neurons.h5ad')

/mnt/rds/genetics01/JinLab/dmp131/.conda/envs/higlue/lib/python3.9/site-packages/anndata/__init__.py:51: FutureWarning: `anndata.read` is deprecated, use `anndata.read_h5ad` instead. `ad.read` will be removed in mid 2024.
  warnings.warn(


In [2]:
rna.layers['counts']

<12531x29738 sparse matrix of type '<class 'numpy.float32'>'
	with 45673533 stored elements in Compressed Sparse Row format>

In [3]:
rna.obs['celltype'].value_counts()

celltype
L2/3     3701
L4       2245
Sst      1414
Vip      1357
L5       1253
Pvalb    1123
L6        996
Ndnf      442
Name: count, dtype: int64

In [4]:
import numpy as np

n_cells = 1000

subsample_idxs = np.random.choice(rna.n_obs, n_cells, replace=False)

rna_subsample = rna[subsample_idxs]
rna_subsample.write('brain_autism_PFC_neurons_subsample.h5ad')

The scHi-C data is input the exact same as our SCORE package (https://github.com/JinLabBioinfo/SCORE) which documents how to obtain a .scool file and a metadata reference file.

To run HiGLUE, we provide the scRNA-seq as an AnnData object and a corresponding GTF file, the scHi-C data as a .scool file and metadata reference file, and several other arguments for preprocessing and training. The arguments are divided in half between HiGLUE specific arguments at the start, and scHi-C specific preprocessing arguments which are passed to the SCORE package when loading the scHi-C dataset. 

There are two main functionalities in the `higlue.py` script:

* `--preprocess` will load the scRNA and scHi-C data, map them to appropriate genomic coordinates, and construct the guidance graph for linking the features.

* `--train` will look for the preprocessed scRNA, scHi-C, and guidance graph, and then train the HiGLUE model

Note that these can be run simultaneously, but training is only possible if the preprocessed inputs are present. Some important arguments are:

* `--n_strata` indicates the max interaction distance considered (same usage as in `SCORE`, we recommend at least interactions within 1Mb)

* `--n_genes` number of highly-variable genes to use when embedding scRNA and constructing guidance graph (in practice we find ~5000 to work best)

* `--min_count` interactions with fewer total counts across the dataset are dropped

* `--gene_list` provide extra genes which will always be included in guidance graph even if they are not in the set of HVGs

* `--loop_q` quantile cutoff for including a pseudo-bulk interaction in the guidance graph (we recommend ~0.99 which will usually give ~200-300k interactions)

* `--filter_strata` fraction of interactions at each strata to keep (we recommend ~0.05 at resolutions 50kb or higher, otherwise this parameter makes little difference)

In [12]:
! python ../higlue.py --rna_file /mnt/jinstore/JinLab01/dmp131/scRNA/brain_autism_PFC_neurons.h5ad \
						--gtf /mnt/jinstore/JinLab01/dmp131/scRNA/hg19_genes.gtf \
						--use_2d \
						--offset 1 \
						--n_strata 10 \
						--filter_strata 0.05 \
						--n_genes 500 \
						--min_count 3 \
						--gene_list NDNF SV2C GAD1 \
						--loop_q 0.995 \
						--preprocess \
						SCORE \
						--dset pfc_neurons \
						--scool /mnt/jinstore/JinLab02/dmp131/HiGLUE/scools/pfc_all_dist_200kb.scool \
						--reference /mnt/jinstore/JinLab02/dmp131/HiGLUE/data/pfc_neurons_ref \
						--resolution 200kb \
						--min_depth 50000;

Parsing SCORE args...
['--dset', 'pfc_neurons', '--scool', '/mnt/jinstore/JinLab02/dmp131/HiGLUE/scools/pfc_all_dist_200kb.scool', '--reference', '/mnt/jinstore/JinLab02/dmp131/HiGLUE/data/pfc_neurons_ref', '--resolution', '200kb', '--min_depth', '50000']
Dataset: pfc
Total Cells: 586
Cells before filtering: 586
Cells before filtering by chr: 586
reference depth < min_depth : 1026
chr_reads < chr_length : 24
not filtered : 562
Reference file with filtering criteria saved to data/pfc_filtered_ref
L2/3: 172
L5: 51
Vip: 73
Ndnf: 51
L6: 32
L4: 49
Pvalb: 45
Sst: 89
Filtering top loops in each chromosome...
100%|███████████████████████████████████████████| 24/24 [01:13<00:00,  3.06s/it]
       bin1_id  bin2_id  count      rank
0            3        4   1137  0.561188
1            4        5   3575  0.902674
2            4        6   1357  0.635410
3            5        6   2669  0.835054
4            6        7   3546  0.901320
...        ...      ...    ...       ...
25234    14035    14036

Now we can run the same command but with the `--train` argument to load the preprocessed data and train the model. There are additional training specific arguments such as `--use_attn` to utilize local attention layers in the scHi-C feature composition and `--max_epochs` to determine how long to train the model. You can also provide `--wandb` to track the training metrics.

Note that you can do preprocessing and training in the same command, just provide both `--preprocess` and `--train` and the model will be trained directly after preprocessing is done.

In [ ]:
! python ../higlue.py --rna_file /mnt/jinstore/JinLab01/dmp131/scRNA/brain_autism_PFC_neurons.h5ad \
						--gtf /mnt/jinstore/JinLab01/dmp131/scRNA/hg19_genes.gtf \
						--use_2d \
						--offset 1 \
						--n_strata 10 \
						--filter_strata 0.05 \
						--n_genes 500 \
						--min_count 3 \
						--gene_list NDNF SV2C GAD1 \
						--loop_q 0.995 \
						--train \
						--use_attn \
						--max_epochs 1 \
						SCORE \
						--dset pfc_neurons \
						--scool /mnt/jinstore/JinLab02/dmp131/HiGLUE/scools/pfc_all_dist_200kb.scool \
						--reference /mnt/jinstore/JinLab02/dmp131/HiGLUE/data/pfc_neurons_ref \
						--resolution 200kb \
						--min_depth 50000;

During training you can monitor the `tmp_imgs` directory to see how the embedding space is progressing.

After training, you will have a new `scanpy` object containing both datasets embedded in a shared subspace: